In [9]:
# require: pip install pulp
import pulp as pl

# --- Données ---

# Shifts possibles (4h consécutives)
shifts = ["8-12","9-13","10-14","11-15","12-16","13-17",
          "14-18","15-19","16-20","17-21","18-22","19-23","20-0"]

# Heures de la journée (8h à 23h)
hours  = list(range(8,24))

# Couverture des shifts : quelles heures sont couvertes par chaque shift
shift_hours = {
    "8-12": [8,9,10,11],
    "9-13": [9,10,11,12],
    "10-14":[10,11,12,13],
    "11-15":[11,12,13,14],
    "12-16":[12,13,14,15],
    "13-17":[13,14,15,16],
    "14-18":[14,15,16,17],
    "15-19":[15,16,17,18],
    "16-20":[16,17,18,19],
    "17-21":[17,18,19,20],
    "18-22":[18,19,20,21],
    "19-23":[19,20,21,22],
    "20-0": [20,21,22,23],
}

# delta[(J,h)] = 1 si le shift J couvre l'heure h, 0 sinon
delta = {}
for J in shifts:
    for h in hours:
        delta[(J,h)] = 1 if (J in shift_hours and h in shift_hours[J]) else 0

# Coût par shift
C = { "8-12": 40, "9-13": 40, "10-14":40, "11-15":40, "12-16":40, "13-17":40, 
      "14-18":40, "15-19":44, "16-20":48, "17-21":52,"18-22":56,
      "19-23":56,"20-0":56
}

# Demande par heure
D = {8: 500, 9: 600, 10: 400, 11: 700, 12: 1500, 13: 1800, 14: 1200, 15: 6000,
     16: 800, 17: 9000, 18: 2000, 19: 22000, 20: 1800, 21: 1500, 22: 300, 23: 200}

# Nombre total de livreurs disponibles
N = 5000

# Capacité par livreur par heure (dictionnaire)
capacity_per_hour = {8: 9, 9: 11, 10: 10, 11: 8, 12: 7, 13: 9, 14: 8, 15: 13, 16: 12, 17: 11, 18: 10, 19: 8, 20: 9, 21: 7, 22: 4, 23: 3}


# --- Modèle ---
prob = pl.LpProblem("ShiftScheduling", pl.LpMinimize)

# Variables : nombre de livreurs par shift
y = {J: pl.LpVariable(f"y_{J}", lowBound=0, cat="Integer") for J in shifts}

# Slack par heure pour garantir faisabilité
u = {h: pl.LpVariable(f"unmet_{h}", lowBound=0, cat="Integer") for h in hours}

# Objectif : coût total des shifts + grosse pénalité pour le manque de colis
BIGPENALTY = 1000
prob += pl.lpSum(C.get(J,0)*y[J] for J in shifts) + BIGPENALTY * pl.lpSum(u[h] for h in hours)

# Contraintes : couverture de la demande par heure avec capacité variable
for h in hours:
    prob += pl.lpSum(delta[(J,h)] * y[J] * capacity_per_hour[h] for J in shifts) + u[h] >= D[h], f"cover_hour_{h}"

# Limite du nombre total de livreurs
prob += pl.lpSum(y[J] for J in shifts) <= N, "max_total_drivers"

# Bornes supérieures par shift (impossible d'affecter plus que N livreurs)
for J in shifts:
    prob += y[J] <= N

# --- Résolution ---
prob.solve(pl.PULP_CBC_CMD(msg=True))

# --- Résultats ---
print("Status:", pl.LpStatus[prob.status])
print("Objective:", pl.value(prob.objective))
print("\nNombre de livreurs par shift :")
for J in shifts:
    print(f"{J} -> {int(pl.value(y[J]))}")

print("\nColis livrés par heure :")
for h in hours:
    delivered = sum(delta[(J,h)] * y[J].varValue * capacity_per_hour[h] for J in shifts)
    print(f"Hour {h}: {delivered} colis (Demande: {D[h]})")


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/homebrew/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/ry/yqwb4m311v78_wyjtn9yym7m0000gn/T/59666206ed50471f82635c192911db5b-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/ry/yqwb4m311v78_wyjtn9yym7m0000gn/T/59666206ed50471f82635c192911db5b-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 35 COLUMNS
At line 217 RHS
At line 248 BOUNDS
At line 278 ENDATA
Problem MODEL has 30 rows, 29 columns and 94 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 158876 - 0.00 seconds
Cgl0003I 0 fixed, 16 tightened bounds, 0 strengthened rows, 0 substitutions
Cgl0004I processed model has 17 rows, 29 columns (29 integer (0 of which binary)) and 81 elements
Cutoff increment increased from 1e-05 to 3.9999
Cbc0012I Integer solution of 158984 fou